In [ ]:
# ============================================
# SESSION 5
# LASSO, RIDGE & ELASTIC NET REGULARIZATION
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score


# ============================================
# 1. LOAD DATASET
# ============================================

CSV_PATH = "/content/placement_predict_50k Dataset.csv"

df = pd.read_csv(
    CSV_PATH,
    on_bad_lines="skip",
    engine="python"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


# ============================================
# 2. SELECT NUMERICAL FEATURES
# ============================================

numeric_df = df.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).copy()

print("\nNumerical columns:")
print(numeric_df.columns.tolist())


# ============================================
# 3. REMOVE MISSING VALUES
# ============================================

numeric_df = numeric_df.dropna()

print("\nShape after removing missing values:")
print(numeric_df.shape)


# ============================================
# 4. SELECT TARGET
# ============================================

# Use CGPA as regression target
if "CGPA" in numeric_df.columns:

    target = "CGPA"

    X = numeric_df.drop(columns=[target])
    y = numeric_df[target]

else:

    # If CGPA is not available,
    # use the last numerical column as target
    target = numeric_df.columns[-1]

    X = numeric_df.drop(columns=[target])
    y = numeric_df[target]

print("\nTarget variable:", target)


# ============================================
# 5. TRAIN-VALIDATION SPLIT
# ============================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])


# ============================================
# 6. IMPUTE MISSING VALUES
# ============================================

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)


# ============================================
# 7. FEATURE SCALING
# ============================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)


# ============================================
# 8. REGULARIZATION STRENGTHS
# ============================================

C_values = [0.01, 0.1, 1, 10, 100]

# Convert C to alpha
# Higher alpha = stronger regularization
alpha_values = [1 / c for c in C_values]

print("\nRegularization strengths:")
print(alpha_values)


# ============================================
# 9. COMPARE LASSO, RIDGE & ELASTIC NET
# ============================================

results = []


for alpha in alpha_values:

    # ----------------------------------------
    # LASSO
    # ----------------------------------------

    lasso = Lasso(
        alpha=alpha,
        max_iter=10000
    )

    lasso.fit(X_train, y_train)

    lasso_pred = lasso.predict(X_val)

    lasso_acc = r2_score(
        y_val,
        lasso_pred
    )

    lasso_nonzero = np.sum(
        lasso.coef_ != 0
    )


    # ----------------------------------------
    # RIDGE
    # ----------------------------------------

    ridge = Ridge(
        alpha=alpha
    )

    ridge.fit(X_train, y_train)

    ridge_pred = ridge.predict(X_val)

    ridge_acc = r2_score(
        y_val,
        ridge_pred
    )

    ridge_nonzero = np.sum(
        ridge.coef_ != 0
    )


    # ----------------------------------------
    # ELASTIC NET
    # ----------------------------------------

    elastic = ElasticNet(
        alpha=alpha,
        l1_ratio=0.5,
        max_iter=10000
    )

    elastic.fit(X_train, y_train)

    elastic_pred = elastic.predict(X_val)

    elastic_acc = r2_score(
        y_val,
        elastic_pred
    )

    elastic_nonzero = np.sum(
        elastic.coef_ != 0
    )


    # ----------------------------------------
    # STORE RESULTS
    # ----------------------------------------

    results.append({

        "C": 1 / alpha,

        "Lasso_val_acc": lasso_acc,

        "Ridge_val_acc": ridge_acc,

        "ElasticNet_val_acc": elastic_acc,

        "Lasso_nonzero": lasso_nonzero,

        "Ridge_nonzero": ridge_nonzero,

        "ElasticNet_nonzero": elastic_nonzero

    })


# ============================================
# 10. RESULTS DATAFRAME
# ============================================

results_df = pd.DataFrame(results)

print("\n============================================")
print("REGULARIZATION COMPARISON")
print("============================================")

print(results_df.to_string(index=False))

Dataset loaded successfully!
Shape: (50000, 32)

Numerical columns:
['StudentID', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly', 'Salary Package']

Shape after removing missing values:
(32877, 24)

Target variable: CGPA

Training samples: 26301
Validation samples: 6576

Regularization strengths:
[100.0, 10.0, 1.0, 0.1, 0.01]

REGULARIZATION COMPARISON
     C  Lasso_val_acc  Ridge_val_acc  ElasticNet_val_acc  Lasso_nonzero  Ridge_nonzero  ElasticNet_nonzero
  0.01      -0.000021       0.942331           -0.000021              0             23                   0
  0.10      -0.000021       0.942251           -0.000021              0             23                   0
  1.00       0.468579       0.94